<a href="https://colab.research.google.com/github/souvikkai/souvik-ai-pm-portfolio/blob/main/day17-llm-judge-eval/DAY17_LLM_as_a_judge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install groq anthropic -q

import anthropic
import os
from groq import Groq
from google.colab import userdata

anthropic_client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

print("Clients initialized")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 7.0 MB/s eta 0:00:00
Clients initialized


In [2]:
RESUME = """
SOUVIK KUNDU
Silicon PM | ASIC Lifecycle (RTL→GDS) × AI Data Center Infrastructure

SUMMARY
Semiconductor and AI PM with 9+ years of hands-on silicon experience — from RTL design and EDA toolchain optimization to process node PDK delivery and hardware NPI. Delivered Intel 18A ASIC PDKs directly to NVIDIA and Microsoft, working alongside datapath architects and RTL teams throughout the full silicon lifecycle: RTL → synthesis → place-and-route → signoff → GDS tape-out → customer bring-up. Deep fluency in PPA tradeoffs, on-chip interconnect constraints, and memory hierarchy behavior — the same decisions that define networking ASIC performance. Currently driving $270M in AI data center design wins at ONSEMI and collaborating with NVIDIA on 800VDC Blackwell GPU rack architecture. Deployed production CNN inference system with full MLOps stack on Vertex AI.

CORE COMPETENCIES
Silicon & ASIC: Full RTL→GDS lifecycle · PDK delivery (NVIDIA, Microsoft) · EDA ecosystem (Synopsys, Cadence, Siemens) · PPA optimization · on-chip interconnect · memory hierarchy (SRAM, HBM) · process node architecture · Intel 18A · NPI · AEC-Q101
AI Networking & Systems: AI data center infrastructure · GPU cluster architecture · NVIDIA Blackwell/Spectrum-X · 800VDC power architecture · high-bandwidth interconnects · latency/throughput/bandwidth tradeoffs · scale-up and scale-out systems

EXPERIENCE
Senior PM, Silicon Power Division | ONSEMI 02/2025–Present
- AI Data Center: Leading NPI for SiC power semiconductor dies into AI data center PSU applications. Secured $270M design wins with AI data center PSU suppliers. Collaborating with NVIDIA on 800VDC power architecture for Blackwell GPU racks. Translating system-level power architecture requirements into device-level specs — the same silicon-to-system translation required for networking ASIC products.
- Production AI System: Led 15-person team deploying CNN classifier for inline semiconductor production inspection (Vertex AI, 8 GPUs) — 99% recall, 4%→100% lot coverage, defect rate 10%→0%. Owned confidence-threshold routing, HITL queue logic, biweekly retraining pipeline maintaining >90% Average Precision.
- EV Platform: Secured Volkswagen design win ($350M platform); led joint team improving EV range ~10%.

PM, Silicon Power Division | ONSEMI 10/2023–01/2025
- 0→1 Silicon Commercialization: Launched semiconductor die sales business from zero — scaled to ~5% of company revenue. Standardized OEM designs saving $4M/year.

PM, Client Computing Group | Intel 09/2022–09/2023
- Edge Platform: Reduced enterprise deployment 24 weeks→2 weeks; tiered pricing drove 20% pilot adoption lift; won 4 pilots (~$70M lifetime revenue).

PM, Design Enablement Group | Intel 01/2021–08/2022
- Intel 18A ASIC PDK → NVIDIA & Microsoft: Owned product delivery of Intel's most advanced process node ASIC PDK. Worked directly with NVIDIA and Microsoft architecture and design teams through the full flow: DRC/LVS rule decks, standard cell libraries, timing models, place-and-route constraints, and signoff collateral. Expanded EDA compatibility across Cadence, Siemens, and Synopsys — 25% adoption increase, 50% reduction in PDK iteration cycles, 10% improvement in customer design productivity.
- Full Silicon Lifecycle Fluency: Deep hands-on collaboration with RTL design, synthesis, physical design, and verification teams across multiple process nodes. Built product intuition for timing closure, power grid constraints, interconnect congestion.

Senior Software Engineer, Design Enablement Group | Intel 06/2016–12/2020
- Developed and optimized EDA automation infrastructure across Intel process nodes; improved PPA metrics for leading-edge silicon portfolio.

EDUCATION
MBA — UCLA Anderson School of Management 2023
M.S., Electrical Engineering — University of Cincinnati 2016
"""

JD = """
Product Manager – Networking Silicon (AI Infrastructure)

Overview
Building high-performance infrastructure for next-generation AI workloads. Focus on systems, silicon, and networking. Technical depth, execution speed, and ownership. Problems where performance and scale are critical.

Role sits at intersection of networking protocols and hardware architecture. Define and deliver networking silicon for large-scale AI systems, translating workload behavior into efficient, high-performance hardware.

Core Responsibilities
- Own product lifecycle from early architecture definition through silicon delivery and production
- Translate AI workload traffic patterns into silicon-level features (buffering, scheduling, flow control, telemetry)
- Drive execution across full development path: requirements → architecture → RTL → validation → bring-up
- Ensure alignment between silicon capabilities and system-level design (topology, latency, bandwidth constraints)
- Balance tradeoffs across power, performance, and area to meet aggressive system targets

Core Requirements
- Strong technical background in networking and hardware systems (deeply technical PM role)
- Deep understanding of packet processing pipelines
- Experience working with high-performance interconnects or cluster-scale networking systems
- Proven experience contributing to or leading a full silicon lifecycle (concept through production)
- Ability to map system-level behavior and traffic patterns to hardware-level design decisions

Nice to Have
- Experience with AI/ML infrastructure, accelerator clusters, or high-performance compute systems
- Familiarity with ASIC or networking silicon design workflows
- Background working closely with architecture, RTL, and validation teams

Screening Question
Walk me through how a packet moves through a networking ASIC pipeline, and where performance bottlenecks typically show up.
"""

print("Resume and JD loaded")
print(f"Resume length: {len(RESUME)} chars")
print(f"JD length: {len(JD)} chars")

Resume and JD loaded
Resume length: 3823 chars
JD length: 1899 chars


In [3]:
GENERATOR_SYSTEM_PROMPT = """You are an expert career coach helping candidates prepare for technical PM interviews.

Given a resume and job description, generate exactly 5 talking points the candidate should use in their interview.

Rules:
- Each talking point MUST be grounded in specific experience from the resume
- Do NOT invent or extrapolate experience not explicitly stated in the resume
- Each talking point should directly address a requirement in the job description
- Include specific numbers, outcomes, or named technologies from the resume
- Format: numbered list, 2-3 sentences each

Output format:
1. [Talking point]
2. [Talking point]
3. [Talking point]
4. [Talking point]
5. [Talking point]"""

GENERATOR_USER_PROMPT = f"""Resume:
{RESUME}

Job Description:
{JD}

Generate 5 interview talking points."""

In [4]:
print("Generating with Claude Haiku...")

haiku_response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=1000,
    system=GENERATOR_SYSTEM_PROMPT,
    messages=[{"role": "user", "content": GENERATOR_USER_PROMPT}]
)

haiku_output = haiku_response.content[0].text
haiku_input_tokens = haiku_response.usage.input_tokens
haiku_output_tokens = haiku_response.usage.output_tokens
haiku_cost = (haiku_input_tokens * 1.0 / 1_000_000) + (haiku_output_tokens * 5.0 / 1_000_000)

print("HAIKU OUTPUT:")
print(haiku_output)
print(f"\nTokens — Input: {haiku_input_tokens}, Output: {haiku_output_tokens}")
print(f"Cost: ${haiku_cost:.5f}")

Generating with Claude Haiku...
HAIKU OUTPUT:
1. **Full Silicon Lifecycle Ownership on Advanced Process Nodes**: I've owned the complete RTL→GDS lifecycle for Intel 18A ASIC PDKs delivered directly to NVIDIA and Microsoft, working hands-on with RTL design, synthesis, place-and-route, and signoff teams. This end-to-end silicon delivery experience—from architecture constraints through production bring-up—maps directly to the full development path this role requires, and I've shipped at scale across multiple EDA tools (Synopsys, Cadence, Siemens).

2. **System-to-Silicon Translation on High-Bandwidth Infrastructure**: I'm currently collaborating with NVIDIA on 800VDC power architecture for Blackwell GPU racks and securing $270M in AI data center design wins, which requires translating system-level power and performance requirements into device-level silicon specs. This same silicon-to-system translation—understanding how cluster topology, latency, and bandwidth constraints drive hardware 

In [5]:
print("Generating with Llama 3.1 8B on Groq...")

llama_response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    max_tokens=1000,
    messages=[
        {"role": "system", "content": GENERATOR_SYSTEM_PROMPT},
        {"role": "user", "content": GENERATOR_USER_PROMPT}
    ]
)

llama_output = llama_response.choices[0].message.content
llama_input_tokens = llama_response.usage.prompt_tokens
llama_output_tokens = llama_response.usage.completion_tokens
llama_cost = (llama_input_tokens * 0.05 / 1_000_000) + (llama_output_tokens * 0.08 / 1_000_000)

print("LLAMA OUTPUT:")
print(llama_output)
print(f"\nTokens — Input: {llama_input_tokens}, Output: {llama_output_tokens}")
print(f"Cost: ${llama_cost:.6f}")

Generating with Llama 3.1 8B on Groq...
LLAMA OUTPUT:
1. **Experience Driving Full Silicon Lifecycle**: In my previous role at ONSEMI, I led the NPI for SiC power semiconductor dies into AI data center PSU applications, securing $270M design wins. I believe this experience in managing the full silicon lifecycle, from early architecture definition through silicon delivery and production, will enable me to effectively drive the product development for the Networking Silicon product. In particular, I successfully worked with a 15-person team to deploy a CNN classifier for inline semiconductor production inspection, resulting in 99% recall and 0% defect rate.

2. **Packet Processing Pipelines**: Throughout my career, I have gained strong technical expertise in packet processing pipelines, especially with regards to networking ASIC design. In my previous role at Intel, I worked closely with architecture and design teams to deliver Intel's most advanced process node ASIC PDK, which included 

In [7]:
JUDGE_SYSTEM_PROMPT = """You are an expert evaluator assessing interview talking points generated by an AI.

You will evaluate each talking point on 4 dimensions. For each dimension, return PASS or FAIL with one sentence of reasoning.

Evaluation dimensions:
1. FAITHFULNESS: Is the talking point grounded in information explicitly stated in the resume? FAIL if it introduces experience, skills, or outcomes not present in the resume.
2. JD_RELEVANCE: Does the talking point directly address a specific requirement or responsibility in the job description?
3. SPECIFICITY: Does the talking point include concrete evidence — specific numbers, named technologies, or measurable outcomes from the resume?
4. INTERVIEW_UTILITY: Would this talking point genuinely help the candidate in a real interview for this role?

Output format — return valid JSON only:
{
  "talking_points": [
    {
      "number": 1,
      "text": "first 50 chars of the talking point...",
      "faithfulness": {"verdict": "PASS" or "FAIL", "reasoning": "one sentence"},
      "jd_relevance": {"verdict": "PASS" or "FAIL", "reasoning": "one sentence"},
      "specificity": {"verdict": "PASS" or "FAIL", "reasoning": "one sentence"},
      "interview_utility": {"verdict": "PASS" or "FAIL", "reasoning": "one sentence"}
    }
  ]
}"""

def build_judge_prompt(model_name, talking_points, resume, jd):
    return f"""Evaluate these 5 talking points generated by {model_name}.

RESUME:
{resume}

JOB DESCRIPTION:
{jd}

TALKING POINTS TO EVALUATE:
{talking_points}

Return JSON evaluation only."""

In [9]:
import json

print("Judging Haiku outputs with Claude Sonnet...")

haiku_judge_response = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=2000,
    system=JUDGE_SYSTEM_PROMPT,
    messages=[{
        "role": "user",
        "content": build_judge_prompt("Claude Haiku", haiku_output, RESUME, JD)
    }]
)

haiku_judge_text = haiku_judge_response.content[0].text

try:
    haiku_judge = json.loads(haiku_judge_text)
    print("Haiku evaluation parsed successfully")
except:
    clean = haiku_judge_text.replace("```json", "").replace("```", "").strip()
    haiku_judge = json.loads(clean)
    print("Haiku evaluation parsed after cleaning")

print(json.dumps(haiku_judge, indent=2))

Judging Haiku outputs with Claude Sonnet...
Haiku evaluation parsed after cleaning
{
  "talking_points": [
    {
      "number": 1,
      "text": "Full Silicon Lifecycle Ownership on Advanced Process N",
      "faithfulness": {
        "verdict": "PASS",
        "reasoning": "All claims \u2014 Intel 18A PDK delivery to NVIDIA and Microsoft, RTL/synthesis/P&R/signoff collaboration, and multi-tool EDA coverage (Synopsys, Cadence, Siemens) \u2014 are explicitly stated in the resume."
      },
      "jd_relevance": {
        "verdict": "PASS",
        "reasoning": "Directly addresses the core requirement of 'proven experience contributing to or leading a full silicon lifecycle (concept through production)' and the full development path responsibility."
      },
      "specificity": {
        "verdict": "PASS",
        "reasoning": "Names Intel 18A process node, specific customers (NVIDIA, Microsoft), specific EDA vendors, and specific flow stages (RTL, synthesis, place-and-route, signoff).

In [10]:
print("Judging Llama outputs with Claude Sonnet...")

llama_judge_response = anthropic_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=2000,
    system=JUDGE_SYSTEM_PROMPT,
    messages=[{
        "role": "user",
        "content": build_judge_prompt("Llama 3.1 8B", llama_output, RESUME, JD)
    }]
)

llama_judge_text = llama_judge_response.content[0].text

try:
    llama_judge = json.loads(llama_judge_text)
    print("Llama evaluation parsed successfully")
except:
    clean = llama_judge_text.replace("```json", "").replace("```", "").strip()
    llama_judge = json.loads(clean)
    print("Llama evaluation parsed after cleaning")

print(json.dumps(llama_judge, indent=2))

Judging Llama outputs with Claude Sonnet...
Llama evaluation parsed after cleaning
{
  "talking_points": [
    {
      "number": 1,
      "text": "Experience Driving Full Silicon Lifecycle: In my p",
      "faithfulness": {
        "verdict": "FAIL",
        "reasoning": "The talking point conflates two separate experiences \u2014 the NPI for SiC power dies and the CNN classifier deployment \u2014 implying the CNN work was part of the silicon lifecycle management, which misrepresents the resume."
      },
      "jd_relevance": {
        "verdict": "PASS",
        "reasoning": "The JD explicitly requires proven experience contributing to or leading a full silicon lifecycle from concept through production."
      },
      "specificity": {
        "verdict": "PASS",
        "reasoning": "Includes specific figures: $270M design wins, 15-person team, 99% recall, and 0% defect rate."
      },
      "interview_utility": {
        "verdict": "FAIL",
        "reasoning": "The conflation of unre

In [11]:
dimensions = ["faithfulness", "jd_relevance", "specificity", "interview_utility"]

def score_model(judge_result):
    scores = {d: 0 for d in dimensions}
    for tp in judge_result["talking_points"]:
        for d in dimensions:
            if tp[d]["verdict"] == "PASS":
                scores[d] += 1
    return scores

haiku_scores = score_model(haiku_judge)
llama_scores = score_model(llama_judge)

print("=" * 60)
print("RESULTS: Claude Haiku vs Llama 3.1 8B")
print("Judge: Claude Sonnet 4")
print("=" * 60)
print(f"\n{'Dimension':<20} {'Haiku':>10} {'Llama':>10}")
print("-" * 42)
for d in dimensions:
    print(f"{d:<20} {haiku_scores[d]:>8}/5 {llama_scores[d]:>8}/5")

haiku_total = sum(haiku_scores.values())
llama_total = sum(llama_scores.values())
print("-" * 42)
print(f"{'TOTAL':<20} {haiku_total:>8}/20 {llama_total:>8}/20")

print(f"\n{'=' * 60}")
print("COST COMPARISON")
print(f"{'=' * 60}")
print(f"Haiku cost per run:  ${haiku_cost:.5f}")
print(f"Llama cost per run:  ${llama_cost:.6f}")
if haiku_cost > 0 and llama_cost > 0:
    print(f"Cost ratio: Haiku is {haiku_cost/llama_cost:.1f}x more expensive than Llama")

print(f"\n{'=' * 60}")
print("PM INSIGHT")
print(f"{'=' * 60}")
winner = "Haiku" if haiku_total >= llama_total else "Llama 3.1 8B"
print(f"Quality winner: {winner} ({max(haiku_total, llama_total)}/20 vs {min(haiku_total, llama_total)}/20)")
print(f"Cost winner: {'Llama 3.1 8B' if llama_cost < haiku_cost else 'Haiku'}")
print(f"\nKey question: Is the quality difference worth the cost difference?")

RESULTS: Claude Haiku vs Llama 3.1 8B
Judge: Claude Sonnet 4

Dimension                 Haiku      Llama
------------------------------------------
faithfulness                5/5        2/5
jd_relevance                5/5        4/5
specificity                 5/5        4/5
interview_utility           4/5        1/5
------------------------------------------
TOTAL                      19/20       11/20

COST COMPARISON
Haiku cost per run:  $0.00414
Llama cost per run:  $0.000108
Cost ratio: Haiku is 38.3x more expensive than Llama

PM INSIGHT
Quality winner: Haiku (19/20 vs 11/20)
Cost winner: Llama 3.1 8B

Key question: Is the quality difference worth the cost difference?


In [12]:
print("=" * 60)
print("FAITHFULNESS FAILURES — Did any model hallucinate?")
print("=" * 60)

for model_name, judge_result in [("Haiku", haiku_judge), ("Llama 3.1 8B", llama_judge)]:
    failures = [
        tp for tp in judge_result["talking_points"]
        if tp["faithfulness"]["verdict"] == "FAIL"
    ]
    if failures:
        print(f"\n{model_name} — {len(failures)} faithfulness failure(s):")
        for f in failures:
            print(f"  Talking point {f['number']}: {f['text'][:80]}...")
            print(f"  Reason: {f['faithfulness']['reasoning']}")
    else:
        print(f"\n{model_name} — No faithfulness failures. All talking points grounded in resume.")

print("\n" + "=" * 60)
print("METRICS CLOSE")
print("=" * 60)
print("Primary metric: Faithfulness pass rate")
print(f"  Haiku: {haiku_scores['faithfulness']}/5")
print(f"  Llama: {llama_scores['faithfulness']}/5")
print("\nGuardrail metrics:")
print(f"  JD Relevance — Haiku: {haiku_scores['jd_relevance']}/5, Llama: {llama_scores['jd_relevance']}/5")
print(f"  Cost per run — Haiku: ${haiku_cost:.5f}, Llama: ${llama_cost:.6f}")
print(f"  Interview Utility — Haiku: {haiku_scores['interview_utility']}/5, Llama: {llama_scores['interview_utility']}/5")
print("\nFailure mode: Judge inflates faithfulness scores because both judge")
print("and generator share world knowledge about 'good PM experience' —")
print("catches claims that sound plausible but aren't in the resume.")
print("Mitigation: spot-check any PASS on claims involving specific technical")
print("experience (packet processing, NoC design) against the actual resume text.")

FAITHFULNESS FAILURES — Did any model hallucinate?

Haiku — No faithfulness failures. All talking points grounded in resume.

Llama 3.1 8B — 3 faithfulness failure(s):
  Talking point 1: Experience Driving Full Silicon Lifecycle: In my p...
  Reason: The talking point conflates two separate experiences — the NPI for SiC power dies and the CNN classifier deployment — implying the CNN work was part of the silicon lifecycle management, which misrepresents the resume.
  Talking point 2: Packet Processing Pipelines: Throughout my career,...
  Reason: The resume does not mention packet processing pipelines at all; claiming 'strong technical expertise in packet processing pipelines' introduces a skill not stated anywhere in the resume.
  Talking point 4: Silicon-Level Features: In my production AI system r...
  Reason: The talking point falsely equates software ML pipeline features (confidence-threshold routing, HITL queue logic) with silicon-level hardware features like buffering, scheduling